<a href="https://colab.research.google.com/github/manish-gajria/LLM_engineering_MG/blob/main/MG_Week_3_exercise_Synthetic_data.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Create meeting minutes from an Audio file

I downloaded some Denver City Council meeting minutes and selected a portion of the meeting for us to transcribe. You can download it here:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing

If you'd rather work with the original data, the HuggingFace dataset is [here](https://huggingface.co/datasets/huuuyeah/meetingbank) and the audio can be downloaded [here](https://huggingface.co/datasets/huuuyeah/MeetingBank_Audio/tree/main).

The goal of this product is to use the Audio to generate meeting minutes, including actions.

For this project, you can either use the Denver meeting minutes, or you can record something of your own!

## Please note:

When you run the pip installs in the first cell below, you might get this error - it can be safely ignored - it sounds quite severe, but it doesn't seem to affect anything else in this project!


> ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.



In [1]:
!pip install -q gradio requests torch bitsandbytes transformers sentencepiece accelerate openai httpx==0.27.2

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.4/76.4 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.1/54.1 MB 18.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 322.9/322.9 kB 25.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 118.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 93.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
!pip install -U bitsandbytes

In [2]:
# imports

import os
import requests
from IPython.display import Markdown, display, update_display
from openai import OpenAI
from google.colab import drive
from huggingface_hub import login
from google.colab import userdata
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
import torch
import gradio as gr

In [3]:
# Constants

AUDIO_MODEL = "whisper-1"
LLAMA = "meta-llama/Meta-Llama-3.1-8B-Instruct"

In [4]:
# New capability - connect this Colab to my Google Drive
# See immediately below this for instructions to obtain denver_extract.mp3

drive.mount("/content/drive")
audio_filename = "/content/drive/My Drive/LLMs/denver_extract.mp3"

Mounted at /content/drive


# Download denver_extract.mp3

You can either use the same file as me, the extract from Denver city council minutes, or you can try your own..

If you want to use the same as me, then please download my extract here, and put this on your Google Drive:  
https://drive.google.com/file/d/1N_kpSojRR5RYzupz6nqM8hMSoEF_R7pU/view?usp=sharing


In [5]:
# Sign in to HuggingFace Hub

hf_token = userdata.get('HF_TOKEN')
login(hf_token, add_to_git_credential=True)

In [6]:
# Sign in to OpenAI using Secrets in Colab

openai_api_key = userdata.get('OPENAI_API_KEY')
openai = OpenAI(api_key=openai_api_key)

In [7]:
def constructmessage(user_prompt):
  system_message = "You are an assistant that generates synthetic data sets for the purposes of business testing"
  messages = [
    {"role": "system", "content": system_message},
    {"role": "user", "content": user_prompt}
  ]
  return messages

In [9]:
# messages = constructmessage("Please produce a data set of 10 product SKUs for an online hardware store. Include SKU, description, price and weight")
# print(messages)

[{'role': 'system', 'content': 'You are an assistant that generates synthetic data sets for the purposes of business testing'}, {'role': 'user', 'content': 'Please produce a data set of 10 product SKUs for an online hardware store. Include SKU, description, price and weight'}]


In [8]:
quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [11]:
tokenizer = AutoTokenizer.from_pretrained(LLAMA)
tokenizer.pad_token = tokenizer.eos_token
inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
streamer = TextStreamer(tokenizer)
model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00001-of-00004.safetensors:   0%|          | 0.00/4.98G [00:00<?, ?B/s]

model-00004-of-00004.safetensors:   0%|          | 0.00/1.17G [00:00<?, ?B/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an assistant that generates synthetic data sets for the purposes of business testing<|eot_id|><|start_header_id|>user<|end_header_id|>

Please produce a data set of 10 product SKUs for an online hardware store. Include SKU, description, price and weight<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here's a sample data set for 10 product SKUs:

| **SKU** | **Description** | **Price** | **Weight (lbs)** |
| --- | --- | --- | --- |
| 12345 | 10-Inch Tape Measure | $4.99 | 0.25 |
| 67890 | 18-Volt Cordless Drill | $99.99 | 3.50 |
| 34567 | 12-Inch Level | $24.99 | 1.50 |
| 90123 | 100-Foot 14-Gauge Extension Cord | $14.99 | 4.00 |
| 45678 | 1-Inch Socket Set | $49.99 | 2.00 |
| 98765 | 3-Inch Pneumatic Stapler | $69.99 | 2.25 |
| 11111 | 20-Inch Wrench Set | $39.99 | 5.00 |
| 22222 | 1/4-Inch Impact Driver | $129.99 | 3.00 |
| 33333 | 24-Inch Stud F

In [12]:
response = tokenizer.decode(outputs[0])

In [13]:
display(Markdown(response))

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an assistant that generates synthetic data sets for the purposes of business testing<|eot_id|><|start_header_id|>user<|end_header_id|>

Please produce a data set of 10 product SKUs for an online hardware store. Include SKU, description, price and weight<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here's a sample data set for 10 product SKUs:

| **SKU** | **Description** | **Price** | **Weight (lbs)** |
| --- | --- | --- | --- |
| 12345 | 10-Inch Tape Measure | $4.99 | 0.25 |
| 67890 | 18-Volt Cordless Drill | $99.99 | 3.50 |
| 34567 | 12-Inch Level | $24.99 | 1.50 |
| 90123 | 100-Foot 14-Gauge Extension Cord | $14.99 | 4.00 |
| 45678 | 1-Inch Socket Set | $49.99 | 2.00 |
| 98765 | 3-Inch Pneumatic Stapler | $69.99 | 2.25 |
| 11111 | 20-Inch Wrench Set | $39.99 | 5.00 |
| 22222 | 1/4-Inch Impact Driver | $129.99 | 3.00 |
| 33333 | 24-Inch Stud Finder | $29.99 | 1.00 |
| 44444 | 50-Foot Measuring Wheel | $59.99 | 3.50 |

This data set includes a mix of tools and hardware products, with varying prices and weights. The SKU numbers are randomly generated and may not be sequential or unique in a real-world scenario.<|eot_id|>

In [14]:
def meetingminutes(audio_filename):
  transcription=audio_transcribe(audio_filename)
  system_message = "You are an assistant that produces minutes of meetings from transcripts, with summary, key discussion points, takeaways and action items with owners, in markdown."
  user_prompt = f"Below is an extract transcript of a council meeting. Please write minutes in markdown, including a summary with attendees, location and date; discussion points; takeaways; and action items with owners.\n{transcription}"
  messages = [
      {"role": "system", "content": system_message},
      {"role": "user", "content": user_prompt}
  ]
  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
  outputs = model.generate(inputs, max_new_tokens=2000)
  response=tokenizer.decode(outputs[0])
  system,delimiter,minutes=response.partition("<|start_header_id|>assistant<|end_header_id|>")
  return minutes

In [9]:
def generatedata(userprompt):
  dataset=""
  messages = constructmessage(userprompt)
  tokenizer = AutoTokenizer.from_pretrained(LLAMA)
  tokenizer.pad_token = tokenizer.eos_token
  inputs = tokenizer.apply_chat_template(messages, return_tensors="pt").to("cuda")
  streamer = TextStreamer(tokenizer)
  model = AutoModelForCausalLM.from_pretrained(LLAMA, device_map="auto", quantization_config=quant_config)
  outputs = model.generate(inputs, max_new_tokens=2000, streamer=streamer)
  dataset = tokenizer.decode(outputs[0])
  return dataset

In [ ]:
with gr.Blocks() as ui:
  with gr.Row():
    display_main = gr.Textbox(label="Data Set")
  with gr.Row():
    datarequest = gr.Textbox(label="Enter user prompt for data set generation")

  # Do entry with file input
    def do_entry (userprompt):
      response = generatedata(userprompt)
      print (response)
      return response

    datarequest.submit(do_entry, inputs=[datarequest], outputs=[display_main])

ui.launch(inbrowser=True,debug=True)

It looks like you are running Gradio on a hosted a Jupyter notebook. For the Gradio app to work, sharing must be enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://d2f3a9d3fb5fbdf7ec.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


tokenizer_config.json:   0%|          | 0.00/55.4k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/855 [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:128001 for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 Jul 2024

You are an assistant that generates synthetic data sets for the purposes of business testing<|eot_id|><|start_header_id|>user<|end_header_id|>

Produce a synthetic or made up data set with 10 SKUs for a women's clothing store - show SKU number, item description, current stock in store and price<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Here's a synthetic data set for a women's clothing store with 10 SKUs:

1. **SKU 001**: Women's Casual T-Shirt
   - **Description**: Soft, breathable casual t-shirt with a crew neck and short sleeves.
   - **Current Stock**: 250 pieces
   - **Price**: $19.99

2. **SKU 002**: Women's Summer Dress
   - **Description**: Lightweight, floral print sundress with a V-neck and adjustable straps.
   - **Current Stock**: 150 pieces
   - **Price**: $34.99

3. **SKU 003**: Women's Fleece Jacket
   - **Description**: Cozy, water-re

# Student contribution

Student Emad S. has made this powerful variation that uses `TextIteratorStreamer` to stream back results into a Gradio UI, and takes advantage of background threads for performance! I'm sharing it here if you'd like to take a look at some very interesting work. Thank you, Emad!

https://colab.research.google.com/drive/1Ja5zyniyJo5y8s1LKeCTSkB2xyDPOt6D

## Alternative implementation

Class student Youssef has contributed this variation in which we use an open-source model to transcribe the meeting Audio.

Thank you Youssef!

In [17]:
AUDIO_MODEL = "openai/whisper-medium"
speech_model = AutoModelForSpeechSeq2Seq.from_pretrained(AUDIO_MODEL, torch_dtype=torch.float16, low_cpu_mem_usage=True, use_safetensors=True)
speech_model.to('cuda')
processor = AutoProcessor.from_pretrained(AUDIO_MODEL)

pipe = pipeline(
    "automatic-speech-recognition",
    model=speech_model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch.float16,
    device='cuda',
)

NameError: name 'AutoModelForSpeechSeq2Seq' is not defined

In [ ]:
# Use the Whisper OpenAI model to convert the Audio to Text
result = pipe(audio_filename)

In [ ]:
transcription = result["text"]
print(transcription)